# Day 4：MCP 协议入门

MCP（Model Context Protocol）是 Anthropic 推出的 Agent 工具协议标准。

把它比作 USB 协议：USB 出现之前，每个外设都有自己的接口；USB 出现之后，一个接口兼容所有设备。
MCP 对 AI Agent 的意义是一样的——**统一 LLM 与外部工具/数据源的通信方式**。

今天的学习路线：
1. MCP 是什么 → 核心思想与三大原语
2. Client/Server 架构 → stdio 和 SSE 两种通信方式
3. MCP vs LangChain Tool → 协议标准 vs 代码绑定
4. 动手跑通官方 filesystem MCP 示例
5. 再接入一个 git MCP server

## 一、MCP 是什么

### 问题背景

在 MCP 出现之前，AI Agent 接入工具有一个严重的问题——**每个 LLM 框架都有一套自己的工具定义方式**：

| 框架 | 工具定义方式 |
|------|------------|
| LangChain | `@tool` 装饰器 + Pydantic BaseModel |
| OpenAI SDK | `tools` 参数 + JSON Schema |
| Anthropic SDK | `tools` 参数 + JSON Schema |
| LlamaIndex | `FunctionTool` 类 |

如果你想写一个"文件读取"工具，你需要为每个框架各写一遍适配代码。

MCP 的目标是：**你写一次工具，所有框架都能用**。

### 核心思想：工具即服务

MCP 把工具抽象为三个"原语"（Primitives）：

| 原语 | 含义 | 类比 |
|------|------|------|
| **Tools** | 可调用的函数（文件读写、数据库查询） | REST API 的 POST 请求 |
| **Resources** | 可读取的数据源（文件内容、数据库记录） | REST API 的 GET 请求 |
| **Prompts** | 预定义的 Prompt 模板，可带参数 | 可复用的对话模板 |

**Model Context Protocol** 这个名字的含义：
- **Model**：LLM 模型
- **Context**：模型需要的上下文（数据、工具、指令）
- **Protocol**：协议——标准化的通信方式

简单说：MCP 定义了"模型如何获取和使用外部信息"的标准协议。

## 二、Client/Server 架构

MCP 采用经典的 Client-Server 架构：

```
┌─────────────┐     MCP 协议      ┌─────────────┐
│  MCP Client  │ ◄──────────────► │  MCP Server  │
│  (LLM 应用)   │    stdio / SSE   │  (工具/数据)  │
└─────────────┘                   └─────────────┘
```

- **MCP Client**：集成在 LLM 应用中，负责发起工具调用请求。比如 Claude Desktop、你的 LangGraph Agent。
- **MCP Server**：暴露工具/资源的服务端。可以是本地进程（stdio）或远程服务（SSE）。

### 两种通信方式

| 方式 | 原理 | 适用场景 |
|------|------|---------|
| **stdio** | 通过标准输入输出通信，server 作为子进程运行 | 本地工具（文件操作、shell） |
| **SSE** | 通过 HTTP + Server-Sent Events 通信 | 远程服务（数据库、API 网关） |

**启动一个 MCP Server（stdio 模式）的本质：**
```bash
# MCP Client 启动子进程，通过 stdin/stdout 交换 JSON 消息
python my_mcp_server.py
```

这跟你用 `subprocess.Popen` 启动一个进程然后用 pipe 通信是同一个原理。

### MCP 通信协议详解

MCP 使用 JSON-RPC 2.0 作为消息格式。一次完整的工具调用流程如下：

```
Client                          Server
  │                                │
  │  1. initialize                 │  Client 发握手请求
  │ ─────────────────────────────► │
  │                                │
  │  2. initialized (capabilities) │  Server 返回能力列表
  │ ◄───────────────────────────── │
  │                                │
  │  3. tools/list                 │  Client 询问：你能提供哪些工具？
  │ ─────────────────────────────► │
  │                                │
  │  4. [Tool1, Tool2, ...]        │  Server 返回工具列表
  │ ◄───────────────────────────── │
  │                                │
  │  5. tools/call {name, args}    │  Client 调用指定工具
  │ ─────────────────────────────► │
  │                                │
  │  6. tool result                │  Server 返回执行结果
  │ ◄───────────────────────────── │
```

注意第 3 步：**动态工具发现**——Client 不需要预先知道 Server 有哪些工具，
只要连接上，Server 就会告诉 Client 自己有什么能力。
这是 MCP 相比 LangChain Tool 最大的优势。

## 三、MCP vs LangChain Tool：根本区别

很多人问"MCP 和 LangChain Tool 有什么区别"。答案是：**根本不是同一个层次的东西**。

| 维度 | LangChain Tool | MCP |
|------|---------------|-----|
| **层次** | 代码级别的函数绑定 | 协议级别的标准 |
| **耦合度** | 与 LangChain 生态强耦合 | 与任何框架无关 |
| **发现机制** | 需要手动 import 并注册 | 运行时自动发现（tools/list） |
| **跨框架** | 只能在 LangChain 中用 | OpenAI/Anthropic/LangChain 都能用 |
| **数据类型** | 只有工具（Tool） | 工具 + 资源 + Prompt（三大原语） |
| **部署方式** | 内嵌在 Python 代码中 | 独立的进程或服务 |

**打个比方：**
- LangChain Tool 像是自己写的 Python 函数，只能在你的项目里用
- MCP 像是把工具打包成了 Docker 容器，谁都可以通过标准接口调用

**MCP 的杀手级场景：**
一家公司有一个内部数据库查询工具，用 MCP 封装后：
- Claude Desktop 可以直接接入
- 你的 LangGraph Agent 可以直接接入
- 同事的 OpenAI SDK 项目也可以直接接入

不需要为每个框架各写一遍适配。

## 四、动手：安装 MCP SDK 并跑通 filesystem 示例

首先安装 MCP Python SDK：

```bash
uv pip install mcp
```

然后我们来写一个最小的 MCP Server 和 Client，理解它的工作原理。

In [3]:
from importlib.metadata import version
version("mcp")


'1.27.2'

### 4.1 写一个最小的 MCP Server

下面实现一个"天气查询"MCP Server。虽然功能简单，但包含了 MCP Server 的全部核心要素：工具注册、参数 Schema、执行逻辑。

In [4]:
# ===== 最小 MCP Server：天气查询 =====
# 这个代码通常放在单独的 .py 文件中，通过 stdio 方式运行
# 为了方便演示，我们在 notebook 中展示完整逻辑

from mcp.server import Server, NotificationOptions
from mcp.server.stdio import stdio_server
import mcp.types as types
import asyncio

# 1. 创建 Server 实例
server = Server("weather-server")

# 2. 注册工具列表处理器——当 Client 调用 tools/list 时触发
@server.list_tools()
async def handle_list_tools() -> list[types.Tool]:
    """告诉 Client 我能提供哪些工具"""
    return [
        types.Tool(
            name="get_weather",
            description="查询指定城市的天气信息",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如 北京、上海、Tokyo"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "温度单位",
                        "default": "celsius"
                    }
                },
                "required": ["city"]
            }
        )
    ]

# 3. 注册工具调用处理器——当 Client 调用 tools/call 时触发
@server.call_tool()
async def handle_call_tool(
    name: str,
    arguments: dict
) -> list[types.TextContent]:
    """执行工具并返回结果"""
    if name == "get_weather":
        city = arguments.get("city")
        unit = arguments.get("unit", "celsius")
        # 模拟天气数据（实际项目中这里会调用真实 API）
        weather_data = {
            "城市": city,
            "温度": "25°C" if unit == "celsius" else "77°F",
            "天气": "晴",
            "湿度": "60%"
        }
        return [types.TextContent(
            type="text",
            text=str(weather_data)
        )]
    
    raise ValueError(f"未知工具: {name}")


print("MCP Server 定义完成！")
print("注意：上述代码需要在单独的进程中通过 stdio 方式运行")
print("实际使用时，用 asyncio.run(main()) 启动 server")

MCP Server 定义完成！
注意：上述代码需要在单独的进程中通过 stdio 方式运行
实际使用时，用 asyncio.run(main()) 启动 server


### 4.2 写一个最小的 MCP Client

Client 的作用是连接 Server，自动发现工具，然后调用它们。

In [ ]:
# ===== 最小 MCP Client =====
# Client 通过 stdio 启动 Server 子进程，然后发送 JSON-RPC 消息

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import asyncio


async def run_mcp_client():
    """连接 MCP Server 并调用工具"""
    
    # 定义如何启动 Server（这里以外部的 python 脚本为例）
    server_params = StdioServerParameters(
        command="python",           # 启动命令
        args=["weather_server.py"], # 脚本参数
    )
    
    # 建立 stdio 连接
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. 初始化握手
            await session.initialize()
            print("✅ 握手成功！")
            
            # 2. 自动发现工具（Client 不需要预先知道有什么工具）
            tools = await session.list_tools()
            print(f"✅ 发现 {len(tools.tools)} 个工具:")
            for tool in tools.tools:
                print(f"   - {tool.name}: {tool.description}")
            
            # 3. 调用工具
            result = await session.call_tool("get_weather", {"city": "北京"})
            print(f"✅ 工具调用结果: {result.content[0].text}")

# asyncio.run(run_mcp_client())  # 实际运行时取消注释
print("Client 代码定义完成")
print("核心流程：connect → initialize → list_tools → call_tool")
print("关键是第 2 步 list_tools：运行时动态发现，不需要预先 import 工具定义")

### 4.3 使用 langchain-mcp-adapters

上面是原生的 MCP SDK 用法。但在 LangGraph 项目中，更方便的方式是用 `langchain-mcp-adapters`，
它能把 MCP 工具直接转成 LangChain Tool，无缝集成到你的 Agent 中。

```bash
uv pip install langchain-mcp-adapters
```

In [ ]:
# ===== 用 langchain-mcp-adapters 将 MCP 工具转为 LangChain Tool =====
# 这是最实用的方式——MCP Server 写好之后，几行代码就能变成 LangChain Tool

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.sessions import StdioConnection

# 注意：langchain-mcp-adapters 0.x 版本使用 TypedDict 定义连接配置
# StdioConnection 需要 transport='stdio' 字段来指定通信方式

# 定义一个 MCP Server 的配置
# 实际使用时，这里可以配置多个 MCP Server
client = MultiServerMCPClient(
    connections={
        "filesystem": StdioConnection(
            transport="stdio",
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", "/path/to/allowed/dir"],
        )
    }
)

# get_tools() 是同步方法（不是 async），自动从所有 Server 聚合工具
# tools = client.get_tools()
#
# 然后就可以像普通 LangChain Tool 一样用了：
# llm_with_tools = llm.bind_tools(tools)

print("langchain-mcp-adapters 的核心逻辑：")
print("1. 启动 MCP Server 子进程（stdio）")
print("2. 发送 tools/list 获取工具列表")
print("3. 把每个 MCP Tool 转成 LangChain BaseTool")
print("4. LLM 调用工具时，adapter 把调用转成 MCP 的 tools/call 请求")
print("5. 返回结果自动转成 LangChain ToolMessage")
print()
print("新版本变化：")
print("- connections 参数使用 StdioConnection / SSEConnection 等 TypedDict")
print("- get_tools() 改为同步方法（不再需要 await）")
print("- 支持 tool_interceptors 中间件拦截工具调用")

## 五、实战：接入 git MCP Server

Git 是每个开发者都要用的工具。有了 git MCP Server，你的 Agent 就能读取代码仓库信息：
查看提交历史、读取文件变更、分析代码结构。

社区已有开源的 git MCP Server，下面演示如何配置和使用。

In [ ]:
# ===== 配置 git MCP Server（配置示例） =====
# 使用社区开源的 git MCP server
# 安装：uv pip install mcp-server-git （假设已发布到 PyPI）
# 或从 GitHub 安装

# MultiServerMCPClient 可以同时管理多个 MCP Server
# 每个 Server 提供一组工具，Client 自动聚合所有工具

from langchain_mcp_adapters.client import MultiServerMCPClient

# 同时接入多个 MCP Server
multi_client_config = {
    # Server 1：文件系统工具
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/path/to/allowed/dir"],
        "transport": "stdio",
    },
    # Server 2：Git 工具
    "git": {
        "command": "uvx",
        "args": ["mcp-server-git", "--repository", "/path/to/your/repo"],
        "transport": "stdio",
    },
}

# client = MultiServerMCPClient(multi_client_config)
# tools = await client.get_tools()  # 自动聚合所有 Server 的工具

print("多 MCP Server 聚合示例")
print("\n配置的 Server:")
for name, cfg in multi_client_config.items():
    print(f"  - {name}: {cfg['command']} {' '.join(cfg['args'])}")
print("\n关键能力：Client 自动发现所有 Server 的工具，无需手动注册")

### 理解 MCP Server 的工具注册流程

MCP Server 的核心工作是"注册工具"。下面用代码拆解这个流程：

In [ ]:
# ===== MCP Server 工具注册的完整流程（代码拆解）=====

from mcp.server import Server
import mcp.types as types

# 创建一个 MCP Server
server = Server("my-agent-tools")

# ===== 步骤1：注册工具元数据（list_tools） =====
@server.list_tools()
async def list_tools() -> list[types.Tool]:
    """
    当 Client 调用 tools/list 时返回工具列表
    
    每个工具包含：
    - name: 工具的唯一标识
    - description: 工具的功能描述（会注入到 LLM Prompt）
    - inputSchema: JSON Schema 格式的参数定义
    """
    return [
        types.Tool(
            name="read_code",
            description="读取代码文件内容，支持指定行号范围",
            inputSchema={
                "type": "object",
                "properties": {
                    "file_path": {"type": "string", "description": "文件路径"},
                    "start_line": {"type": "integer", "description": "起始行号（从1开始）"},
                    "end_line": {"type": "integer", "description": "结束行号"}
                },
                "required": ["file_path"]
            }
        ),
        types.Tool(
            name="git_log",
            description="查看 Git 提交历史",
            inputSchema={
                "type": "object",
                "properties": {
                    "max_count": {"type": "integer", "description": "最多显示的提交数量", "default": 10},
                    "author": {"type": "string", "description": "按作者过滤"}
                }
            }
        ),
    ]


# ===== 步骤2：注册工具执行逻辑（call_tool） =====
@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    """
    当 Client 调用 tools/call 时执行具体的工具逻辑
    
    参数：
    - name: Client 要调用的工具名
    - arguments: Client 传入的参数
    
    返回值：TextContent 列表
    """
    if name == "read_code":
        file_path = arguments["file_path"]
        start = arguments.get("start_line", 1)
        end = arguments.get("end_line")
        
        with open(file_path, "r") as f:
            lines = f.readlines()
            result = "".join(lines[start-1:end])
        
        return [types.TextContent(type="text", text=result)]
    
    elif name == "git_log":
        import subprocess
        cmd = ["git", "log", f"-{arguments.get('max_count', 10)}", "--oneline"]
        result = subprocess.run(cmd, capture_output=True, text=True)
        return [types.TextContent(type="text", text=result.stdout)]
    
    else:
        return [types.TextContent(type="text", text=f"错误：未知工具 {name}")]


print("MCP Server 工具注册完成")
print("\n核心要点：")
print("1. list_tools() 定义了'有哪些工具'——这是给 LLM 看的菜单")
print("2. call_tool() 定义了'怎么执行工具'——这是实际干活的")
print("3. 两个函数通过工具名 (name) 关联")
print("4. inputSchema 就是给 LLM 的 '使用说明书'")

## 今日总结

MCP 的核心价值可以归纳为一句话：**让工具可以被任何 LLM 框架调用，而不是绑定在某个特定框架上**。

今天掌握了：

1. **MCP 的三大原语** — Tools（可调用函数）、Resources（可读取数据）、Prompts（可复用模板）
2. **Client/Server 架构** — Client 集成在 LLM 应用中，Server 暴露工具/资源
3. **两种通信方式** — stdio（本地进程，最常用）、SSE（远程 HTTP）
4. **工具注册流程** — list_tools（告诉 Client 有哪些工具）→ call_tool（执行工具）
5. **动态工具发现** — Client 不需要预先 import 工具代码，运行时自动发现
6. **langchain-mcp-adapters** — 一行代码把 MCP 工具转成 LangChain Tool

**写在简历上的要点：**
"熟悉 MCP（Model Context Protocol），能够基于 MCP 协议开发可复用的 Agent 工具服务，
支持动态工具发现和多框架兼容（LangGraph/OpenAI/Anthropic）。"

明天学习 Context 与 Memory 管理——让你的 Agent 保持长对话记忆。